# 🏨 Iceberg Property Combined — Explorer Notebook

**Uses**: `pyspark.sql` → Iceberg catalog → `rental_property` + `property_reviews`  
**View**: `property_combined` (inner join on `rental_property.id == property_reviews.gen_id`)  
**Catalog**: Hadoop FileSystem (`./warehouse`)


## 0 · Setup — SparkSession with Iceberg

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Resolve repo root whether notebook lives at repo root or notebooks/
NOTEBOOK_DIR = os.path.abspath('')
if os.path.exists(os.path.join(NOTEBOOK_DIR, 'config.py')):
    PROJECT_DIR = NOTEBOOK_DIR
else:
    PROJECT_DIR = os.path.dirname(NOTEBOOK_DIR)
WAREHOUSE = os.path.join(PROJECT_DIR, 'warehouse')

CATALOG = 'local'
RENTAL_TABLE  = f'{CATALOG}.property_db.rental_property'
REVIEWS_TABLE = f'{CATALOG}.property_db.property_reviews'
COMBINED_VIEW = 'property_combined'

spark = (
    SparkSession.builder
    .appName('IcebergViewer')
    .master('local[*]')
    .config(
        'spark.sql.extensions',
        'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions'
    )
    .config(f'spark.sql.catalog.{CATALOG}',
            'org.apache.iceberg.spark.SparkCatalog')
    .config(f'spark.sql.catalog.{CATALOG}.type', 'hadoop')
    .config(f'spark.sql.catalog.{CATALOG}.warehouse', WAREHOUSE)
    .config(
        'spark.jars.packages',
        'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2'
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('✅ SparkSession ready — Iceberg catalog:', CATALOG)
print('   Warehouse:', WAREHOUSE)
print('   Tables:', RENTAL_TABLE, 'and', REVIEWS_TABLE)


26/03/24 17:45:21 WARN Utils: Your hostname, w3e39 resolves to a loopback address: 127.0.1.1; using 192.168.0.227 instead (on interface enp2s0)
26/03/24 17:45:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/w3e39/Documents/iceberg_writer/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/w3e39/.ivy2/cache
The jars for the packages stored in: /home/w3e39/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ee75f691-2dd8-49fd-8fa3-e3533e4fa315;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 in central
:: resolution report :: resolve 144ms :: artifacts dl 4ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spar

✅ SparkSession ready — Iceberg catalog: local
   Warehouse: /home/w3e39/Documents/iceberg_writer/warehouse
   Tables: local.property_db.rental_property and local.property_db.property_reviews


## 1 · Schema & Row Count

In [2]:
# Load base Iceberg tables
df_rental  = spark.table(RENTAL_TABLE)
df_reviews = spark.table(REVIEWS_TABLE)

# Build combined view (rental + reviews)
df_combined = (
    df_rental.alias('r')
    .join(df_reviews.alias('v'), F.col('r.id') == F.col('v.gen_id'), how='inner')
    .select(
        F.col('r.id').alias('id'),
        F.col('r.feed_provider_id'),
        F.col('r.property_name'),
        F.col('r.property_slug'),
        F.col('r.country_code'),
        F.col('r.currency'),
        F.col('r.usd_price'),
        F.col('r.star_rating'),
        F.col('r.review_score'),
        F.col('r.commission'),
        F.col('r.meal_plan'),
        F.col('r.published'),
        F.col('r.data_quality_flag'),
        F.col('v.property_id'),
        F.col('v.review_id'),
        F.col('v.review_date'),
        F.col('v.review_year'),
        F.col('v.review_individual_score'),
        F.col('v.review_language'),
        F.col('v.review_summary'),
        F.col('v.review_positive'),
        F.col('v.review_negative'),
        F.col('v.reviewer_name'),
        F.col('v.reviewer_country'),
        F.col('v.reviewer_travel_purpose'),
        F.col('v.reviewer_type'),
    )
)

df_combined.createOrReplaceTempView(COMBINED_VIEW)

print(f'Combined rows : {df_combined.count()}')
print(f'Columns       : {len(df_combined.columns)}')
print('Schema:')
df_combined.printSchema()


26/03/24 17:45:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/24 17:45:35 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
[Stage 1:==================================================>    (110 + 4) / 120]

Combined rows : 6272
Columns       : 26
Schema:
root
 |-- id: string (nullable = false)
 |-- feed_provider_id: string (nullable = true)
 |-- property_name: string (nullable = true)
 |-- property_slug: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- usd_price: double (nullable = true)
 |-- star_rating: double (nullable = true)
 |-- review_score: double (nullable = true)
 |-- commission: double (nullable = true)
 |-- meal_plan: string (nullable = true)
 |-- published: boolean (nullable = true)
 |-- data_quality_flag: string (nullable = true)
 |-- property_id: long (nullable = true)
 |-- review_id: long (nullable = true)
 |-- review_date: string (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_individual_score: double (nullable = true)
 |-- review_language: string (nullable = true)
 |-- review_summary: string (nullable = true)
 |-- review_positive: string (nullable = true)
 |-- review_negative: stri

## 2 · Sample Rows

In [3]:
# Register as a temp view so we can use plain SQL
spark.sql(f"""
    SELECT id, property_name, country_code,
           star_rating, review_score, review_year,
           reviewer_name, review_individual_score, data_quality_flag
    FROM   {COMBINED_VIEW}
    LIMIT  10
""").show(truncate=False)


+-----------+---------------------------+------------+-----------+------------+-----------+-------------+-----------------------+-----------------+
|id         |property_name              |country_code|star_rating|review_score|review_year|reviewer_name|review_individual_score|data_quality_flag|
+-----------+---------------------------+------------+-----------+------------+-----------+-------------+-----------------------+-----------------+
|GEN-1509823|Tilford Woods Lodge Retreat|GB          |0.0        |8.9         |2026       |Tabe         |7.0                    |GOOD             |
|GEN-1509823|Tilford Woods Lodge Retreat|GB          |0.0        |8.9         |2026       |Tabe         |7.0                    |GOOD             |
|GEN-1509823|Tilford Woods Lodge Retreat|GB          |0.0        |8.9         |2026       |Tabe         |7.0                    |GOOD             |
|GEN-1509823|Tilford Woods Lodge Retreat|GB          |0.0        |8.9         |2026       |Tabe         |7.0    

## 3 · Reviews per Country


In [4]:
spark.sql(f"""
    SELECT  country_code,
            COUNT(DISTINCT id)          AS properties,
            COUNT(review_id)            AS total_reviews,
            ROUND(AVG(review_individual_score), 2) AS avg_review_score,
            ROUND(AVG(star_rating), 2)  AS avg_stars
    FROM    {COMBINED_VIEW}
    GROUP BY country_code
    ORDER BY total_reviews DESC
""").show(40, truncate=False)


[Stage 7:===================================================>   (113 + 4) / 120]

+------------+----------+-------------+----------------+---------+
|country_code|properties|total_reviews|avg_review_score|avg_stars|
+------------+----------+-------------+----------------+---------+
|GB          |6         |1024         |9.5             |3.0      |
|ID          |3         |960          |7.27            |2.33     |
|HR          |2         |512          |9.0             |3.63     |
|US          |2         |384          |7.67            |3.83     |
|FI          |1         |320          |9.8             |3.0      |
|MX          |3         |320          |8.4             |2.57     |
|DE          |3         |320          |7.4             |3.17     |
|JP          |1         |320          |8.4             |3.0      |
|BE          |1         |320          |9.6             |4.0      |
|AL          |2         |192          |10.0            |0.0      |
|NO          |1         |192          |5.33            |4.0      |
|FR          |3         |64           |8.0             |2.0   

## 4 · Review Volume by Year


In [5]:
spark.sql(f"""
    SELECT  review_year,
            COUNT(*)                         AS total_review_rows,
            ROUND(AVG(review_individual_score), 2) AS avg_score,
            COUNT(DISTINCT country_code)     AS countries_active
    FROM    {COMBINED_VIEW}
    WHERE   review_year IS NOT NULL
    GROUP BY review_year
    ORDER BY review_year
""").show()


[Stage 14:======================================================> (76 + 2) / 78]

+-----------+-----------------+---------+----------------+
|review_year|total_review_rows|avg_score|countries_active|
+-----------+-----------------+---------+----------------+
|       2023|              512|     8.38|               4|
|       2024|              704|     8.82|               5|
|       2025|             1280|      7.4|               7|
|       2026|             2496|     8.72|              10|
+-----------+-----------------+---------+----------------+



## 5 · Top 10 Properties by Average Review Score

In [6]:
spark.sql(f"""
    SELECT  id,
            FIRST(property_name)                       AS name,
            FIRST(country_code)                        AS country,
            FIRST(star_rating)                         AS stars,
            ROUND(AVG(review_individual_score), 2)     AS avg_score,
            COUNT(review_id)                           AS num_reviews
    FROM    {COMBINED_VIEW}
    WHERE   review_id IS NOT NULL
    GROUP BY id
    HAVING  num_reviews >= 2
    ORDER BY avg_score DESC
    LIMIT   10
""").show(truncate=False)


[Stage 21:=======================================================>(77 + 1) / 78]

+------------+----------------------------+-------+-----+---------+-----------+
|id          |name                        |country|stars|avg_score|num_reviews|
+------------+----------------------------+-------+-----+---------+-----------+
|GEN-3138009 |Ash Cottage                 |GB     |5.0  |10.0     |192        |
|GEN-7013181 |Cotswold View 2             |GB     |4.0  |10.0     |128        |
|GEN-15814192|Sky Luxury Apartment Shkoder|AL     |0.0  |10.0     |192        |
|GEN-174022  |Bridget Inn                 |FI     |3.0  |9.8      |320        |
|GEN-8719233 |Lavanda 2                   |HR     |4.0  |9.8      |320        |
|GEN-10521952|Katmoget Cottage            |GB     |4.0  |9.8      |320        |
|GEN-2254251 |Mercure Blankenberge        |BE     |4.0  |9.6      |320        |
|GEN-3437932 |The Old Piggery - Ukc2619   |GB     |4.0  |9.0      |64         |
|GEN-5989584 |Odiyana Bali Retreat        |ID     |3.0  |9.0      |320        |
|GEN-1509823 |Tilford Woods Lodge Retrea

## 6 · Data Quality Distribution

In [7]:
spark.sql(f"""
    SELECT  data_quality_flag,
            COUNT(DISTINCT id) AS properties,
            COUNT(*)            AS rows
    FROM    {COMBINED_VIEW}
    GROUP BY data_quality_flag
""").show()


[Stage 25:=============================================>        (101 + 4) / 120]

+-----------------+----------+----+
|data_quality_flag|properties|rows|
+-----------------+----------+----+
|             GOOD|        42|6272|
+-----------------+----------+----+



## 7 · Pricing & Commercials


In [8]:
# Pricing and commercial signals at property level (deduplicated)
spark.sql(f"""
    WITH property_base AS (
        SELECT id, property_name, country_code, currency, usd_price, commission, meal_plan, published
        FROM {COMBINED_VIEW}
        GROUP BY id, property_name, country_code, currency, usd_price, commission, meal_plan, published
    )
    SELECT country_code, currency,
           COUNT(*) AS properties,
           ROUND(AVG(usd_price), 2) AS avg_usd_price,
           ROUND(MIN(usd_price), 2) AS min_usd_price,
           ROUND(MAX(usd_price), 2) AS max_usd_price
    FROM property_base
    GROUP BY country_code, currency
    ORDER BY properties DESC
""").show(40, truncate=False)

spark.sql(f"""
    WITH property_base AS (
        SELECT id, usd_price, commission, meal_plan, published
        FROM {COMBINED_VIEW}
        GROUP BY id, usd_price, commission, meal_plan, published
    )
    SELECT meal_plan,
           COUNT(*) AS properties,
           ROUND(AVG(usd_price), 2) AS avg_usd_price,
           ROUND(AVG(commission), 4) AS avg_commission,
           SUM(CASE WHEN commission IS NULL THEN 1 ELSE 0 END) AS missing_commission
    FROM property_base
    GROUP BY meal_plan
    ORDER BY properties DESC
""").show(truncate=False)

spark.sql(f"""
    WITH property_base AS (
        SELECT id, property_name, usd_price, commission, published
        FROM {COMBINED_VIEW}
        GROUP BY id, property_name, usd_price, commission, published
    )
    SELECT id, property_name, usd_price, commission, published
    FROM property_base
    ORDER BY usd_price DESC
    LIMIT 10
""").show(truncate=False)


+------------+--------+----------+-------------+-------------+-------------+
|country_code|currency|properties|avg_usd_price|min_usd_price|max_usd_price|
+------------+--------+----------+-------------+-------------+-------------+
|GB          |GBP     |6         |613.62       |384.43       |781.54       |
|ID          |IDR     |3         |88.51        |32.03        |150.99       |
|DE          |EUR     |3         |523.31       |409.61       |708.82       |
|MX          |USD     |3         |198.36       |115.56       |346.5        |
|FR          |EUR     |3         |699.31       |431.31       |1210.47      |
|DK          |DKK     |2         |715.07       |687.58       |742.56       |
|US          |USD     |2         |1011.95      |476.0        |1547.9       |
|HR          |EUR     |2         |649.67       |257.37       |1041.97      |
|AL          |EUR     |2         |180.08       |103.77       |256.39       |
|IT          |EUR     |2         |898.89       |683.12       |1114.66      |

+-----------+----------+-------------+--------------+------------------+
|meal_plan  |properties|avg_usd_price|avg_commission|missing_commission|
+-----------+----------+-------------+--------------+------------------+
|[]         |37        |523.55       |7.6216        |0                 |
|[breakfast]|5         |287.59       |8.472         |0                 |
+-----------+----------+-------------+--------------+------------------+



[Stage 46:=================================================>    (110 + 4) / 120]

+------------+----------------------------------------+---------+----------+---------+
|id          |property_name                           |usd_price|commission|published|
+------------+----------------------------------------+---------+----------+---------+
|GEN-12929721|Vista Pines Retreat                     |1547.9   |6.96      |true     |
|GEN-15890644|Somptueuse maison familiale avec piscine|1210.47  |8.7       |true     |
|GEN-14690442|4 Bedroom Cozy Home In Ragusa           |1114.66  |6.96      |true     |
|GEN-8849628 |2 Bedroom Amazing Home In Dubravica     |1041.97  |6.96      |true     |
|GEN-9453430 |Amazing Apartment In Jezierzany         |883.29   |6.96      |true     |
|GEN-14992597|Rookery Cottage                         |781.54   |6.96      |true     |
|GEN-15158069|2 Bedroom Stunning Home In Byxelkrok    |763.91   |6.96      |true     |
|GEN-7013181 |Cotswold View 2                         |759.29   |6.96      |true     |
|GEN-15889843|Amazing Home In Ørsted With S

## 8 · Reviewer Travel Purpose Breakdown


In [9]:
spark.sql(f"""
    SELECT  reviewer_travel_purpose,
            COUNT(*)                              AS reviews,
            ROUND(AVG(review_individual_score),2) AS avg_score
    FROM    {COMBINED_VIEW}
    WHERE   reviewer_travel_purpose IS NOT NULL
    GROUP BY reviewer_travel_purpose
    ORDER BY reviews DESC
""").show()


[Stage 50:===================================>                    (50 + 4) / 78]

+-----------------------+-------+---------+
|reviewer_travel_purpose|reviews|avg_score|
+-----------------------+-------+---------+
|                leisure|   4160|     8.48|
|               business|    832|     7.77|
+-----------------------+-------+---------+



## 9 · Join Coverage & Review Density


In [10]:
spark.sql(f"""
    SELECT
        COUNT(DISTINCT id) AS properties_total,
        COUNT(DISTINCT CASE WHEN review_id IS NOT NULL THEN id END) AS properties_with_reviews,
        COUNT(DISTINCT CASE WHEN review_id IS NULL THEN id END) AS properties_without_reviews,
        COUNT(review_id) AS total_reviews
    FROM {COMBINED_VIEW}
""").show(truncate=False)

spark.sql(f"""
    SELECT  id,
            COUNT(review_id) AS reviews_per_property
    FROM    {COMBINED_VIEW}
    GROUP BY id
    ORDER BY reviews_per_property DESC
    LIMIT   10
""").show(truncate=False)


+----------------+-----------------------+--------------------------+-------------+
|properties_total|properties_with_reviews|properties_without_reviews|total_reviews|
+----------------+-----------------------+--------------------------+-------------+
|42              |22                     |20                        |4992         |
+----------------+-----------------------+--------------------------+-------------+



[Stage 61:==============================================>       (103 + 4) / 120]

+------------+--------------------+
|id          |reviews_per_property|
+------------+--------------------+
|GEN-10569673|320                 |
|GEN-174022  |320                 |
|GEN-2254251 |320                 |
|GEN-8719233 |320                 |
|GEN-1509823 |320                 |
|GEN-5989584 |320                 |
|GEN-10521952|320                 |
|GEN-242052  |320                 |
|GEN-8856332 |320                 |
|GEN-6831651 |320                 |
+------------+--------------------+



## 10 · Filtered Reviews (GB, 2024)


In [11]:
# Filtered scan over country + review_year
spark.sql(f"""
    SELECT  property_name, reviewer_name,
            review_date, review_individual_score, review_summary
    FROM    {COMBINED_VIEW}
    WHERE   country_code = 'GB'
      AND   review_year  = 2024
    ORDER BY review_individual_score DESC
""").show(truncate=False)


+----------------+-------------+-----------+-----------------------+--------------------------------------------------------------------------------------------------+
|property_name   |reviewer_name|review_date|review_individual_score|review_summary                                                                                    |
+----------------+-------------+-----------+-----------------------+--------------------------------------------------------------------------------------------------+
|Katmoget Cottage|Leanne       |2024-02-15 |10.0                   |Very comfortable cottage with everything you needed to make to a perfect stay.                    |
|Katmoget Cottage|Leanne       |2024-02-15 |10.0                   |Very comfortable cottage with everything you needed to make to a perfect stay.                    |
|Katmoget Cottage|Leanne       |2024-02-15 |10.0                   |Very comfortable cottage with everything you needed to make to a perfect stay.              

## 11 · Reviewer Type vs Star Rating Correlation


In [12]:
spark.sql(f"""
    SELECT  reviewer_type,
            ROUND(AVG(star_rating), 2)             AS avg_stars,
            ROUND(AVG(review_individual_score), 2) AS avg_review_score,
            COUNT(*)                               AS total_reviews
    FROM    {COMBINED_VIEW}
    WHERE   reviewer_type IS NOT NULL
    GROUP BY reviewer_type
    ORDER BY avg_review_score DESC
""").show(truncate=False)


[Stage 67:=============================================>          (64 + 4) / 78]

+--------------------+---------+----------------+-------------+
|reviewer_type       |avg_stars|avg_review_score|total_reviews|
+--------------------+---------+----------------+-------------+
|couple              |2.74     |8.82            |2176         |
|family_with_children|3.47     |8.74            |1216         |
|extended_group      |2.63     |8.25            |512          |
|solo_traveller      |3.06     |7.06            |1088         |
+--------------------+---------+----------------+-------------+



## 13 · Clean Up


In [13]:
spark.stop()
print('SparkSession stopped.')

SparkSession stopped.
